In [1]:
import os
import re
import math
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer,set_seed
from peft import LoraConfig,PeftModel
from datetime import datetime
from datasets import load_dataset,Dataset,DatasetDict
import wandb
from peft import LoraConfig
# TRL/Transformers compatibility guard
try:
    from trl import SFTTrainer, SFTConfig
except Exception:
    import transformers.utils as _tf_utils

    if not hasattr(_tf_utils, "is_liger_kernel_available"):
        _tf_utils.is_liger_kernel_available = lambda: False

    from trl import SFTTrainer, SFTConfig


c:\Users\KRISH\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Constants

BASE_MODEL="meta-llama/Meta-Llama-3.1-8b"
PROJECT_NAME="pricer"
HF_USER="Krishwall"

DATASET_NAME="ed-donner/pricer-data"

MAX_SEQUENCE_LENGTH=182

# Run name for saving the model in the hub
RUN_NAME=f"qlora-{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
PROJECT_RUN_NAME=f"{PROJECT_NAME}_{RUN_NAME}"
HUB_MODEL_NAME=f"{HF_USER}/{PROJECT_RUN_NAME}"

# Hyperparameters for QLoRA
LORA_R=32
LORA_ALPHA=64 # The paper suggests setting alpha to r*2, but you can experiment with this value alpha*lora_a*lora*b
TARGET_MODULES=["q_proj","v_proj","k_proj","o_proj"
                ]
LORA_DROPOUT=0.1
QUANT_4_BIT=True

# Training Hyperparameters

EPOCHS=3
BATCH_SIZE=16
GRADIENT_ACCUMULATION_STEPS=1
LEARNING_RATE= 1e-4
LR_SCHEDULER_TYPE="cosine" # it is often recommended to use a cosine scheduler with a warmup phase for fine-tuning large language models.
# The cosine scheduler helps to gradually decrease the learning rate, which can lead to better convergence and improved performance. 
# The warmup phase allows the model to start with a lower learning rate, which can help stabilize training in the initial stages.
WARMUP_RATIO=0.03
OPTIMIZER="paged_adamw_32bit"

# Admin Config

STEPS=50
SAVE_STEPS=5000
LOG_TO_WANDB=True


In [3]:
hf_token=os.getenv("HF_TOKEN")
wandb_api_key=os.getenv("WAND_API_KEY")


In [4]:
os.environ["WANDB_PROJECT"]=PROJECT_NAME
os.environ["WANDB_LOG_MODEL"]="true" if LOG_TO_WANDB else "false"
os.environ["WANDB_WATCH"]="gradients"


In [5]:
dataset=load_dataset(DATASET_NAME)
train=dataset["train"]
test=dataset["test"]


In [6]:
len(train)

400000

In [7]:
train[0]

{'text': 'How much does this cost to the nearest dollar?\n\nDelphi FG0166 Fuel Pump Module\nDelphi brings 80 years of OE Heritage into each Delphi pump, ensuring quality and fitment for each Delphi part. Part is validated, tested and matched to the right vehicle application Delphi brings 80 years of OE Heritage into each Delphi assembly, ensuring quality and fitment for each Delphi part Always be sure to check and clean fuel tank to avoid unnecessary returns Rigorous OE-testing ensures the pump can withstand extreme temperatures Brand Delphi, Fit Type Vehicle Specific Fit, Dimensions LxWxH 19.7 x 7.7 x 5.1 inches, Weight 2.2 Pounds, Auto Part Position Unknown, Operation Mode Mechanical, Manufacturer Delphi, Model FUEL PUMP, Dimensions 19.7\n\nPrice is $227.00',
 'price': 226.95}

In [10]:
if LOG_TO_WANDB:
    wandb.init(project=PROJECT_NAME, name=PROJECT_RUN_NAME)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: ERROR Invalid API key: API key must have 40+ characters, has 1.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do no

In [11]:
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
else:
    quant_config=BitsAndBytesConfig(load_in_8bit=True)

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL,trust_remote_code=True)
tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="right"

base_model=AutoModelForCausalLM.from_pretrained(
BASE_MODEL,quantization_config=quant_config,device_map="auto"
)
base_model.generation_config.pad_token_id=tokenizer.pad_token_id

print(f"Memory footprint:{base_model.get_memory_footprint() /1e6:.1f} MB")

### DATA COLLATOR

The model needs to predict the tokens after "Price is $"it doesnt need to learn the text
data only take them in context while predictiong the result

In [ ]:

from trl import DataCollatorForCompletionOnlyLM
response_template="Price is $"
collator=DataCollatorForCompletionOnlyLM(response_template,tokenizer=tokenizer) # 


In [ ]:

lora_parameters=LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES
)

# Next,

train_parameters= SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs==EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    eval_strategy="no",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_kwargs=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="text",
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    


)


fine_tuning=SFTTrainer(
    model=base_model,
    train_dataset=train,
    peft_config=lora_parameters,
    tokenizer=tokenizer,
    args=train_parameters,
    data_collator=collator
)

In [ ]:
fine_tuning.train()

fine_tuning.model.push_to_hub(PROJECT_RUN_NAME)
print(f"Saved to the hub: {PROJECT_RUN_NAME}" )


In [ ]:
if LOG_TO_WANDB:
    wandb.finish()